# 面试问题：怎样为 Agent 设计好用的 Tool？Schema、错误、分页、幂等和 Held-out Eval 如何实现？

**一句话回答**：Tool 是确定性系统与非确定性 Agent 的契约。名称和描述明确“做什么、何时用/不用”，输入用严格而语义化的字段，输出只返回完成下一步所需的高信号数据；副作用工具提供 dry-run、幂等键、风险标注和可验证结果。工具质量必须用真实多步任务的 held-out rollout 评测，而不是只跑 API 单测。

本 Notebook 手写工具元数据 lint、JSON Schema 子集校验、结构化错误、分页响应、风险/幂等控制、描述冲突检测、轨迹指标和 schema canary。所有关键代码都有中文注释。


In [ ]:
from dataclasses import dataclass
import hashlib,json,math,re
from collections import Counter

# 受控工具目录用于测试选择冲突和 schema 行为。
SEED149=14901
assert SEED149==14901
assert re.fullmatch(r"[a-z][a-z0-9_]*","tickets_search")
assert len(hashlib.sha256(b"tool").hexdigest())==64


## 1. 名称、description 和边界共同决定可发现性

`jira_search_issues` 比 `search` 更有命名空间；description 写适用场景、术语、不会做什么与必要前置条件。不要把所有 REST endpoint 一比一暴露，也不要让两个工具描述几乎相同却副作用不同。元数据版本应进入 trace。


In [ ]:
@dataclass(frozen=True)
class ToolMeta149:
    name:str; description:str; version:str; side_effect:str
    def __post_init__(self):
        # 工具名限制为稳定的小写命名空间，描述过短无法帮助模型选型。
        if not re.fullmatch(r"[a-z][a-z0-9_]{2,63}",self.name) or len(self.description)<20 or self.side_effect not in {"none","write","destructive"}: raise ValueError("tool_metadata")
meta149=ToolMeta149("tickets_search","按项目和状态搜索工单；只读取，不创建或修改工单。","v1","none")
assert meta149.side_effect=="none"
try: ToolMeta149("Search","短", "v1","none"); raise AssertionError("bad meta")
except ValueError as e: assert str(e)=="tool_metadata"
assert meta149.name.startswith("tickets_")


## 2. 输入字段语义明确，并在执行前严格校验

使用 `project_id` 而不是含糊的 `project`，枚举 sort/status，数值给上下界，默认值由 Host 填充并写入规范化调用。未知字段默认拒绝可暴露模型幻觉；兼容策略按 schema version 明确，不能静默丢弃危险参数。


In [ ]:
SCHEMA149={"type":"object","required":["project_id","query"],"additionalProperties":False,"properties":{"project_id":{"type":"string"},"query":{"type":"string"},"limit":{"type":"integer","minimum":1,"maximum":100},"status":{"type":"string","enum":["open","closed"]}}}
def validate149(value,schema,path="$"):
    # 只实现教学所需 JSON Schema 子集，并返回所有可定位错误。
    errors=[]
    if schema.get("type")=="object":
        if not isinstance(value,dict): return [f"{path}:object_required"]
        for k in schema.get("required",[]):
            if k not in value: errors.append(f"{path}.{k}:required")
        if not schema.get("additionalProperties",True):
            for k in value.keys()-schema.get("properties",{}).keys(): errors.append(f"{path}.{k}:unknown")
        for k,v in value.items():
            if k in schema.get("properties",{}): errors+=validate149(v,schema["properties"][k],f"{path}.{k}")
    elif schema.get("type")=="string" and not isinstance(value,str): errors.append(f"{path}:string_required")
    elif schema.get("type")=="integer":
        if isinstance(value,bool) or not isinstance(value,int): errors.append(f"{path}:integer_required")
        elif not schema.get("minimum",-math.inf)<=value<=schema.get("maximum",math.inf): errors.append(f"{path}:range")
    if "enum" in schema and value not in schema["enum"]: errors.append(f"{path}:enum")
    return errors
assert validate149({"project_id":"p1","query":"timeout","limit":20,"status":"open"},SCHEMA149)==[]
assert "$.limit:range" in validate149({"project_id":"p1","query":"x","limit":999},SCHEMA149)
assert "$.extra:unknown" in validate149({"project_id":"p1","query":"x","extra":1},SCHEMA149)


## 3. 错误告诉 Agent 下一步可执行动作

错误包含稳定 code、retryable、字段路径、简短 message 与必要 remediation；不要把 stack trace、SQL 或 secret 返回模型。认证失败与对象不存在避免过度泄露。Agent 可依据 retryable/required_input 决定重试、改参数或询问用户。


In [ ]:
def tool_error149(code,message,retryable=False,path=None,required_input=None):
    # 结构化字段让循环无需从自然语言猜测错误类型。
    return {"ok":False,"error":{"code":code,"message":message,"retryable":retryable,"path":path,"required_input":required_input}}
err149=tool_error149("INVALID_ARGUMENT","limit 必须在 1 到 100",False,"$.limit")
assert not err149["ok"]
assert err149["error"]["path"]=="$.limit"
assert err149["error"]["retryable"] is False


## 4. 输出按任务过滤、分页并保留 provenance

搜索类工具默认返回 top results、稳定 ID、关键字段、cursor 和 total estimate，不把整个数据库塞进上下文。Agent 可用 ID 调详情工具；response 中标记截断、数据版本和权限过滤。分页 cursor 不接受模型任意拼接内部 offset。


In [ ]:
ROWS149=[{"id":f"T-{i}","title":f"issue {i}","secret":"hidden"} for i in range(7)]
def page149(rows,limit,cursor=0):
    # 只投影 agent 下一步需要的字段，cursor 由工具产生。
    chunk=rows[cursor:cursor+limit]; nxt=cursor+limit if cursor+limit<len(rows) else None
    return {"items":[{"id":x["id"],"title":x["title"]} for x in chunk],"next_cursor":nxt,"truncated":nxt is not None}
page1_149=page149(ROWS149,3)
assert len(page1_149["items"])==3
assert page1_149["next_cursor"]==3
assert all("secret" not in x for x in page1_149["items"])


## 5. 写工具提供 dry-run、幂等键和 postcondition

dry-run 返回规范化 diff、风险与所需审批；commit 使用同一 plan digest 和 idempotency key。Host 验证 capability/approval，工具执行后返回业务对象 ID、版本和可检查状态。重试不能重复创建订单或发送消息。


In [ ]:
commits149={}
def create_ticket149(args,dry_run,key=None):
    # plan digest 把审批与最终规范化参数绑定。
    plan=hashlib.sha256(json.dumps(args,sort_keys=True).encode()).hexdigest()
    if dry_run: return {"ok":True,"plan_digest":plan,"would_create":args}
    if key in commits149:
        if commits149[key]["plan_digest"]!=plan: raise ValueError("idempotency_conflict")
        return commits149[key]
    result={"ok":True,"ticket_id":f"T-{len(commits149)+100}","plan_digest":plan}; commits149[key]=result; return result
plan149=create_ticket149({"title":"bug"},True); first149=create_ticket149({"title":"bug"},False,"k1"); again149=create_ticket149({"title":"bug"},False,"k1")
assert first149==again149
assert first149["plan_digest"]==plan149["plan_digest"]
assert len(commits149)==1


## 6. 检测名字/描述重叠与危险工具混淆

若 `delete_user` 与 `disable_user` 描述都只写“管理用户”，模型很难选择。用 token overlap 作为静态探针，再通过真实 rollout 看误选；危险工具使用显式动词、窄 schema 和独立权限，不靠在 description 末尾写“请小心”。


In [ ]:
def words149(text): return set(re.findall(r"[a-z]+|[\u4e00-\u9fff]+",text.lower()))
def jaccard149(a,b):
    # 静态相似度只做候选告警，最终仍由 held-out tool-use eval 判断。
    A,B=words149(a),words149(b); return len(A&B)/(len(A|B) or 1)
similar149=jaccard149("搜索 工单 项目 状态","搜索 工单 项目 标签"); distinct149=jaccard149("搜索 工单","创建 日历 会议")
assert similar149>distinct149
assert 0<=similar149<=1
assert jaccard149("same","same")==1


## 7. Held-out Eval 覆盖选择、参数、轨迹和最终状态

构造真实多步任务，允许多条合法策略。指标拆为正确工具选择、参数有效率、工具错误、调用数/token/延迟、任务成功和禁止副作用；读取 raw transcript 区分“没选到”“参数错”“输出难理解”。调 description/schema 只用训练集，held-out 防过拟合。


In [ ]:
rollouts149=[{"selected":1,"valid":1,"success":1,"calls":2},{"selected":1,"valid":0,"success":0,"calls":3},{"selected":0,"valid":0,"success":0,"calls":0},{"selected":1,"valid":1,"success":1,"calls":1}]
# 分层指标定位失败发生在选择、参数还是环境结果。
select_rate149=sum(x["selected"] for x in rollouts149)/len(rollouts149); valid_rate149=sum(x["valid"] for x in rollouts149)/sum(x["selected"] for x in rollouts149); success149=sum(x["success"] for x in rollouts149)/len(rollouts149)
assert select_rate149==.75
assert math.isclose(valid_rate149,2/3)
assert success149==.5


## 8. Schema 变更走版本、兼容适配和 canary

新 required 字段、枚举删除和语义变化是 breaking change；保留 v1 adapter 或发布 v2 名称/协议版本。稳定 hash 绑定模型 prompt 中的 schema；小流量 canary 比较任务成功、invalid args 与 token 成本，异常回滚 catalog pointer。


In [ ]:
def schema_digest149(schema):
    # canonical JSON 保证字段顺序变化不导致虚假版本漂移。
    return hashlib.sha256(json.dumps(schema,sort_keys=True,separators=(",",":")).encode()).hexdigest()
schema_v2_149={**SCHEMA149,"properties":{**SCHEMA149["properties"],"sort":{"type":"string","enum":["recent","relevant"]}}}
assert schema_digest149(SCHEMA149)!=schema_digest149(schema_v2_149)
assert len(schema_digest149(SCHEMA149))==64
assert validate149({"project_id":"p1","query":"x","sort":"recent"},schema_v2_149)==[]


## 面试总结

完整设计是：**选择面向 Agent 的合适粒度 → namespace/name/description 写适用与禁用边界 → strict semantic schema → 可行动结构化错误 → 高信号字段+分页+provenance → 写操作 dry-run/plan digest/idempotency/postcondition → 静态检测重叠 → 真实多步 held-out rollout 分层归因 → schema hash/canary/rollback**。API 可调用不代表工具对 Agent 可用。

延伸阅读：[Writing Effective Tools for Agents](https://www.anthropic.com/engineering/writing-tools-for-agents)、[MCP Tools Specification](https://modelcontextprotocol.io/specification/2025-11-25/server/tools)、[Gorilla](https://arxiv.org/abs/2305.15334)。
